# TrustHire — results

Every number below is loaded from `results/` (real runs). Synthetic smoke-test outputs (`results/SYNTHETIC_DEMO`) are deliberately excluded.

In [ ]:
import json, pathlib
import pandas as pd
import matplotlib.pyplot as plt
R = pathlib.Path('..') / 'results'
load = lambda p: json.loads((R / p).read_text()) if (R / p).exists() else None

## Voice anti-spoofing (public data)

In [ ]:
b = load('audio/benchmark.json')
df = pd.DataFrame(b.values())[['dataset','condition','backend','n','auc','eer','tpr_at_fpr1','ece']]
df['eer_%'] = (100*df.eer).round(2); df.sort_values(['dataset','condition','eer'])

In [ ]:
rob = df[df.dataset=='itw'].pivot(index='condition', columns='backend', values='eer_%')
ax = rob.plot.bar(figsize=(8,3.5), logy=True, rot=20); ax.set_ylabel('EER % (log)'); ax.set_title('In-the-Wild: robustness to degradations'); plt.tight_layout()

## Identity

In [ ]:
f = load('identity/face_lfw.json'); s = load('identity/speaker_librispeech.json')
pd.DataFrame([{k: d.get(k) for k in ['benchmark','n','auc','eer','eer_threshold_sim']} for d in (f, s) if d])

## AV sync desync study

In [ ]:
a = load('avsync/desync_study.json')
pd.DataFrame(a['per_shift']).T if a else 'pending'

## Latency (per 3 s window)

In [ ]:
l = load('latency/latency_cpu.json')
pd.Series(l['modules_ms_p50']).sort_values().plot.barh(figsize=(6,3), title=f"CPU window p50 {l['total_ms_p50']:.0f} ms"); plt.tight_layout()

## Custom interview test set (ablations)

In [ ]:
c = load('custom/results.json')
if c and not c['synthetic']:
    rows = [{'config': v['config'], 'fuser': v['fuser'], **{k: v['window'][k] for k in ('auc','eer','tpr_at_fpr5','ece')}} for v in c['ablation'].values()]
    display(pd.DataFrame(rows).sort_values('auc', ascending=False))
else:
    print('pending: record the custom set (docs/DATA_COLLECTION.md) and run `make eval-custom`')